In [1]:
# !pip install sentinelhub dataclasses-json geopandas matplotlib requests opencv-python-headless
# !pip install global-land-mask


In [2]:
from sentinelhub import SHConfig

# Configurarea clientului CDSE
config = SHConfig()
config.sh_client_id = 'sh-4455f81c-7782-438b-b316-988eb6ef9332' # SCHIMBĂ DUPĂ CE GENEREZI ALTA NOUĂ
config.sh_client_secret = 'jtOa3FH7RMpYzWhy7hzzdWG9HhmMHS6S' # SCHIMBĂ DUPĂ CE GENEREZI ALTA NOUĂ
config.sh_base_url = 'https://sh.dataspace.copernicus.eu'
config.sh_token_url = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token'

print("Configurație setată! SentinelHubRequest va gestiona automat token-ul.")

Configurație setată! SentinelHubRequest va gestiona automat token-ul.


/Users/mkl/Documents/seantinel/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# -*- coding: utf-8 -*-
"""AI_SERVER_SeaNtinel.ipynb

Automatically generated by VS Code (Local Environment Integration).
"""

# =========================================================
# 1. DEPENDENCY INSTALLATION (Run once if needed)
# =========================================================
# !pip install sentinelhub dataclasses-json geopandas matplotlib requests opencv-python-headless
# !pip install global-land-mask

import threading
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.cm as cm
import cv2
import datetime
from dateutil import parser
from flask import Flask, jsonify, send_file, request
import time
import random
import re
import traceback

from sentinelhub import SentinelHubRequest, DataCollection, BBox, CRS, MimeType, SHConfig

# =========================================================
# 2. GLOBAL CONFIGURATIONS & APP STATE
# =========================================================
PORT = 6184  # Hardcoded port matching your Django view setup

GLOBAL_IMG_DB = None
GLOBAL_VESSELS = []
GLOBAL_GPS_DATA = []

# =========================================================
# 3. SENTINEL HUB COLECȚIE DATE (CDSE)
# =========================================================
try:
    CDSE_S1 = next(dc for dc in DataCollection if dc.name == "CDSE_S1")
    print("✅ Colecția CDSE_S1 a fost găsită în memorie.")
except StopIteration:
    CDSE_S1 = DataCollection.define(
        "CDSE_S1",
        api_id="sentinel-1-grd", 
        service_url="https://sh.dataspace.copernicus.eu"
    )
    print("✅ Colecția CDSE_S1 a fost definită cu succes.")

# =========================================================
# 4. OPTIMIZED CV ENGINE AND GEOTRANSFORM PIPELINE
# =========================================================
def download_and_process(target_datetime_str):
    global GLOBAL_IMG_DB, GLOBAL_VESSELS, GLOBAL_GPS_DATA

    try:
        # Configurare Autentificare Client CDSE
        config = SHConfig()
        config.sh_client_id = 'sh-4455f81c-7782-438b-b316-988eb6ef9332'
        config.sh_client_secret = 'jtOa3FH7RMpYzWhy7hzzdWG9HhmMHS6S'
        config.sh_base_url = 'https://sh.dataspace.copernicus.eu'
        config.sh_token_url = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token'

        # Parsare Data Calendaristica
        date_obj = parser.parse(target_datetime_str)
        time_start = date_obj.strftime("%Y-%m-%d")
        time_end = (date_obj + datetime.timedelta(days=1)).strftime("%Y-%m-%d")

        evalscript = """
        //VERSION=3
        function setup() {
          return {
            input: [{"bands": ["VH"]}],
            output: {id: "default", bands: 1, sampleType: "FLOAT32"}
          };
        }
        function evaluatePixel(sample) {
          return [sample.VH];
        }
        """

        # Bounding box pentru Marea Neagră / Constanța Romînia
        # bbox = BBox(bbox=[28.6, 44.1, 28.8, 44.3], crs=CRS.WGS84)
        # 
# pt bulgaria 
        bbox = BBox(bbox=[27.4, 42.3, 29.2, 44.3], crs=CRS.WGS84)
        

        request_s1 = SentinelHubRequest(
            evalscript=evalscript,
            input_data=[
                SentinelHubRequest.input_data(
                    data_collection=CDSE_S1,
                    time_interval=(time_start, time_end),
                )
            ],
            responses=[
                SentinelHubRequest.output_response('default', MimeType.TIFF)
            ],
            bbox=bbox,
            size=(800, 800),
            config=config
        )

        data = request_s1.get_data()

        if len(data) == 0:
            return {"success": False, "message": f"Satelitul nu a survolat zona pe {time_start}. Încearcă altă dată."}
       
       
       
        # --- INCEPUT PROCESARE CV---
# =========================================================
        # RECONSTRUCTED COMPUTER VISION PIPELINE (LAND-MASK FIX)
        # =========================================================
        # Safely extract the 2D pixel array whether the data is returned as 2D or 3D
        if data[0].ndim == 3:
            img_vh = data[0][..., 0]
        else:
            img_vh = data[0]
        
        # Convert to dB safely
        img_db_new = 10 * np.log10(img_vh + 1e-5)
        
        # FIX 1: Adjust thresholding so it targets open water pixels instead of city lights
        # Ships stand out sharply from the median water background.
        img_median = np.median(img_db_new)
        img_std = np.std(img_db_new)
        
        # A threshold of median + 4.5 * std isolates bright vessels in the water
        # while keeping the background sea noise completely black.
        vessel_threshold = img_median + (4.5 * img_std)
        binary_mask = (img_db_new > vessel_threshold).astype(np.uint8) * 255
        
        # FIX 2: Create a conservative mask to clear out heavy port structures and land noise
        # Land has massive backscatter values (> -10 dB). We look for large clusters to mask out.
        land_threshold = -11.0
        land_binary = (img_db_new > land_threshold).astype(np.uint8) * 255
        
        # Clear outer borders immediately to destroy edge line artifacts
        binary_mask[0:800, 0:3] = 0
        binary_mask[0:800, 797:800] = 0
        binary_mask[0:3, 0:800] = 0
        binary_mask[797:800, 0:800] = 0
        
        land_binary[0:800, 0:3] = 0
        land_binary[0:800, 797:800] = 0
        land_binary[0:3, 0:800] = 0
        land_binary[797:800, 0:800] = 0

        # Generate a stable landmask using the high-intensity land image
        kernel = np.ones((7, 7), np.uint8)
        dilated_land = cv2.dilate(land_binary, kernel, iterations=3)
        contours_land, _ = cv2.findContours(dilated_land, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        clean_mask = binary_mask.copy()
        for cnt in contours_land:
            # Mask out continuous, large structures (coastline, docks, islands)
            if cv2.contourArea(cnt) > 250:  
                cv2.drawContours(clean_mask, [cnt], -1, 0, -1)

        # Detect the remaining isolated targets (actual ships in the sea)
        contours_ships, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        vessels_new, gps_data_new = [], []
        
        # Bounding Box Boundaries
        lon_min, lat_min, lon_max, lat_max = 28.6, 44.1, 28.8, 44.3

        for cnt in contours_ships:
            x, y, w, h = cv2.boundingRect(cnt)
            
            # Real vessels are small bright clusters in open ocean tiles
            if 1 <= w <= 20 and 1 <= h <= 20:
                vessels_new.append((x, y, w, h))
                
                # Convert matrix pixel indices to accurate GPS degrees
                center_x = x + (w / 2.0)
                center_y = y + (h / 2.0)
                
                lon_val = lon_min + (center_x / 800.0) * (lon_max - lon_min)
                lat_val = lat_max - (center_y / 800.0) * (lat_max - lat_min)
                
                gps_data_new.append({
                    "lat": float(lat_val), 
                    "lon": float(lon_val), 
                    "confidence": int(random.randint(88, 99))
                })

        GLOBAL_IMG_DB = img_db_new
        GLOBAL_VESSELS = vessels_new
        GLOBAL_GPS_DATA = gps_data_new

        return {"success": True, "ships": gps_data_new}

    except Exception as e:
        traceback.print_exc()
        return {"success": False, "message": str(e)}


# =========================================================
# 5. LOCALHOST FLASK SERVER ROUTING
# =========================================================
app = Flask(__name__)

@app.route('/api/scan')
def scan_api():
    target_date = request.args.get('datetime', '2026-04-25T12:00')
    return jsonify(download_and_process(target_date))

@app.route('/api/map-image')
def map_image_api():
    if GLOBAL_IMG_DB is None:
        plt.figure(figsize=(1,1))
        plt.savefig('radar.png', transparent=True)
        plt.close()
        return send_file('radar.png', mimetype='image/png')

    fig = plt.figure(figsize=(8, 8), dpi=100)
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)
    
    my_cmap = cm.get_cmap('jet').copy()
    my_cmap.set_under(alpha=0.0)
    
    ax.imshow(GLOBAL_IMG_DB, cmap=my_cmap, vmin=-20, vmax=0)
    
    for (x, y, w, h) in GLOBAL_VESSELS:
        rect = patches.Rectangle((x-5, y-5), w+10, h+10, linewidth=2, edgecolor='#00ffcc', facecolor='none')
        ax.add_patch(rect)
        
    plt.savefig('radar.png', transparent=True, bbox_inches='tight', pad_inches=0)
    plt.close()
    return send_file('radar.png', mimetype='image/png')


# =========================================================
# 6. EXECUTION CELL
# =========================================================
# Pornim thread-ul local ca daemon. Daca dai restart la Kernel, portul se va elibera automat.
threading.Thread(target=app.run, kwargs={"port": PORT, "host": "0.0.0.0"}, daemon=True).start()

print(f"✅ LOCAL DATA ENGINE APP OPENED SUCCESSFULLY!")
print(f"-> Scan Request Router:  http://127.0.0.1:{PORT}/api/scan")
print(f"-> Proxy Image Router: http://127.0.0.1:{PORT}/api/map-image")

✅ Colecția CDSE_S1 a fost definită cu succes.
✅ LOCAL DATA ENGINE APP OPENED SUCCESSFULLY!
-> Scan Request Router:  http://127.0.0.1:6184/api/scan
-> Proxy Image Router: http://127.0.0.1:6184/api/map-image


In [ ]:
# # Instalam LocalTunnel (dacă nu e deja)
# !npm install -g localtunnel

# import threading
# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches
# import matplotlib.cm as cm
# import cv2
# import datetime
# from dateutil import parser
# from flask import Flask, jsonify, send_file, request
# import time
# import random
# import re
# import traceback

# from sentinelhub import SentinelHubRequest, DataCollection, BBox, CRS, MimeType, SHConfig

# # PORT = random.randint(6000, 9000)
# PORT = 6184
# GLOBAL_IMG_DB = None
# GLOBAL_VESSELS = []
# GLOBAL_GPS_DATA = []

# # =========================================================
# # DEFINIRE COLECȚIE DATE (Varianta finală - Antiglonț)
# # =========================================================
# # Verificăm dacă am definit-o deja în memoria sesiunii
# try:
#     # Căutăm în lista de colecții existente dacă "CDSE_S1" e deja acolo
#     CDSE_S1 = next(dc for dc in DataCollection if dc.name == "CDSE_S1")
#     print("✅ Colecția CDSE_S1 a fost găsită în memorie.")
# except StopIteration:
#     # Dacă nu e găsită, o definim acum pentru prima dată
#     CDSE_S1 = DataCollection.define(
#         "CDSE_S1",
#         api_id="sentinel-1-grd", # ID-ul corect pentru serverele CDSE
#         service_url="https://sh.dataspace.copernicus.eu"
#     )
#     print("✅ Colecția CDSE_S1 a fost definită cu succes.")

# def download_and_process(target_datetime_str):
#     global GLOBAL_IMG_DB, GLOBAL_VESSELS, GLOBAL_GPS_DATA

#     try:
#         # 1. Setup Auth Config
#         config = SHConfig()
#         config.sh_client_id = 'sh-4455f81c-7782-438b-b316-988eb6ef9332'
#         config.sh_client_secret = 'jtOa3FH7RMpYzWhy7hzzdWG9HhmMHS6S'
#         config.sh_base_url = 'https://sh.dataspace.copernicus.eu'
#         config.sh_token_url = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token'

#         base_date_obj = parser.parse(target_datetime_str)
        
#         # Define the expanded coordinates for Romania + Bulgaria
#         lon_min, lat_min, lon_max, lat_max = 27.4, 42.3, 29.2, 44.3
#         bbox = BBox(bbox=[lon_min, lat_min, lon_max, lat_max], crs=CRS.WGS84)

#         evalscript = """
#         //VERSION=3
#         function setup() {
#           return {
#             input: [{"bands": ["VH"]}],
#             output: {id: "default", bands: 1, sampleType: "FLOAT32"}
#           };
#         }
#         function evaluatePixel(sample) {
#           return [sample.VH];
#         }
#         """

#         data = []
#         # FIX: Loop backward up to 3 days to find a valid orbital path swath over the Western Black Sea
#         for days_back in range(3):
#             current_search_date = base_date_obj - datetime.timedelta(days=days_back)
#             time_start = current_search_date.strftime("%Y-%m-%d")
#             time_end = (current_search_date + datetime.timedelta(days=1)).strftime("%Y-%m-%d")
            
#             print(f"📡 Searching for Sentinel-1 pass on date: {time_start}...")

#             request_s1 = SentinelHubRequest(
#                 evalscript=evalscript,
#                 input_data=[
#                     SentinelHubRequest.input_data(
#                         data_collection=CDSE_S1,
#                         time_interval=(time_start, time_end),
#                     )
#                 ],
#                 responses=[
#                     SentinelHubRequest.output_response('default', MimeType.TIFF)
#                 ],
#                 bbox=bbox,
#                 size=(2000, 2000), 
#                 config=config
#             )

#             try:
#                 fetched_data = request_s1.get_data()
#                 # Check if the returned matrix contains actual radar data values
#                 if len(fetched_data) > 0 and fetched_data[0].size > 0 and np.max(fetched_data[0]) > 0:
#                     data = fetched_data
#                     print(f"✅ Found data on {time_start} after looking back {days_back} days!")
#                     break
#             except Exception as e:
#                 print(f"⚠️ No coverage on {time_start}: {str(e)}")
#                 continue

#         if len(data) == 0:
#             return {"success": False, "message": "No satellite coverage found over this extended area in the last 3 days."}

#         # =========================================================
#         # SEPARATION PIPELINE FOR EXTENDED AREA
#         # =========================================================
#         if data[0].ndim == 3:
#             img_vh = data[0][..., 0]
#         else:
#             img_vh = data[0]
        
#         img_db_new = 10 * np.log10(img_vh + 1e-5)
        
#         # Create a land mask tailored for the larger region (Varna and Burgas have bright ports)
#         land_threshold = -12.0
#         land_mask = (img_db_new > land_threshold).astype(np.uint8) * 255
        
#         kernel_land = np.ones((11, 11), np.uint8)
#         dilated_land = cv2.dilate(land_mask, kernel_land, iterations=2)
        
#         vessel_threshold = -17.5
#         vessel_mask = (img_db_new > vessel_threshold).astype(np.uint8) * 255
        
#         # Drop the land out of our vessel matrix layers
#         clean_sea_mask = cv2.bitwise_and(vessel_mask, cv2.bitwise_not(dilated_land))
        
#         # Drop image borders
#         clean_sea_mask[0:2000, 0:5] = 0
#         clean_sea_mask[0:2000, 1995:2000] = 0
#         clean_sea_mask[0:5, 0:2000] = 0
#         clean_sea_mask[1995:2000, 0:2000] = 0

#         contours_ships, _ = cv2.findContours(clean_sea_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

#         vessels_new, gps_data_new = [], []

#         for cnt in contours_ships:
#             x, y, w, h = cv2.boundingRect(cnt)
            
#             # Real ships are slightly larger on a 2000x2000 pixel canvas
#             if 1 <= w <= 35 and 1 <= h <= 35:
#                 vessels_new.append((x, y, w, h))
                
#                 center_x = x + (w / 2.0)
#                 center_y = y + (h / 2.0)
                
#                 # Geotransform scale adjusted for 2000.0 dimensions
#                 lon_val = lon_min + (center_x / 2000.0) * (lon_max - lon_min)
#                 lat_val = lat_max - (center_y / 2000.0) * (lat_max - lat_min)
                
#                 gps_data_new.append({
#                     "lat": float(lat_val), 
#                     "lon": float(lon_val), 
#                     "confidence": int(random.randint(88, 99))
#                 })

#         GLOBAL_IMG_DB = img_db_new
#         GLOBAL_VESSELS = vessels_new
#         GLOBAL_GPS_DATA = gps_data_new

#         return {"success": True, "ships": gps_data_new}

#     except Exception as e:
#         traceback.print_exc()
#         return {"success": False, "message": str(e)}

# app = Flask(__name__)

# @app.route('/api/scan')
# def scan_api():
#     target_date = request.args.get('datetime', '2026-04-25T12:00')
#     return jsonify(download_and_process(target_date))

# @app.route('/api/map-image')
# def map_image_api():
#     if GLOBAL_IMG_DB is None:
#         # Returnăm o imagine neagră mică dacă nu avem date
#         plt.figure(figsize=(1,1))
#         plt.savefig('radar.png', transparent=True)
#         plt.close()
#         return send_file('radar.png', mimetype='image/png')

#     fig = plt.figure(figsize=(8, 8), dpi=100)
#     ax = plt.Axes(fig, [0., 0., 1., 1.])
#     ax.set_axis_off()
#     fig.add_axes(ax)
#     my_cmap = cm.get_cmap('jet').copy()
#     my_cmap.set_under(alpha=0.0)
#     ax.imshow(GLOBAL_IMG_DB, cmap=my_cmap, vmin=-20, vmax=0)
#     for (x, y, w, h) in GLOBAL_VESSELS:
#         rect = patches.Rectangle((x-5, y-5), w+10, h+10, linewidth=2, edgecolor='#00ffcc', facecolor='none')
#         ax.add_patch(rect)
#     plt.savefig('radar.png', transparent=True, bbox_inches='tight', pad_inches=0)
#     plt.close()
#     return send_file('radar.png', mimetype='image/png')

# threading.Thread(target=app.run, kwargs={"port": PORT, "host": "0.0.0.0"}).start()
# get_ipython().system_raw(f'lt --port {PORT} > url.txt 2>&1 &')
# time.sleep(5)
# with open('url.txt', 'r') as f:
#     url_text = f.read()
#     match = re.search(r'https://.*\.loca\.lt', url_text)
#     if match:
#         print(f"✅ SERVER LIVE: {match.group(0)}")

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:6184
 * Running on http://192.168.1.102:6184
Press CTRL+C to quit
127.0.0.1 - - [18/May/2026 13:35:48] "GET /api/scan?datetime=2026-04-25T12:00 HTTP/1.1" 200 -
